# 03 商业 EDA 分析

这一阶段的目标是把清洗后的订单宽表转化成业务洞察。

EDA 不应随机画图，而应围绕业务问题建立指标体系：

1. 平台整体规模和增长表现如何？
2. 收入来自哪些品类、地区和支付方式？
3. 物流履约是否影响客户评分？
4. 哪些业务环节最值得优先优化？

本 Notebook 的输出会保存到 `reports/figures/`，方便后续写 README、报告和 Dashboard。

## 0. 导入依赖与路径


In [ ]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
matplotlib_cache_dir = project_root / ".matplotlib_cache"
matplotlib_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(matplotlib_cache_dir)

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.2f}".format

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

processed_dir, figures_dir


## 1. 读取分析主表

`orders_analysis_base.csv` 是上一阶段构建的一行一订单宽表。后续大部分业务指标都从这张表出发。

In [ ]:
orders = pd.read_csv(processed_dir / "orders_analysis_base.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["purchase_month_dt"] = orders["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
orders["purchase_quarter"] = orders["order_purchase_timestamp"].dt.to_period("Q").astype(str)

orders.shape


## 2. 指标口径

先明确口径，避免分析时混用概念。

- `orders`：订单数，一行一个 `order_id`
- `GMV`：这里用 `payment_total` 表示订单支付总额
- `AOV`：Average Order Value，平均订单支付金额，即 GMV / 订单数
- `product_total`：商品金额，不含运费
- `freight_total`：运费金额
- `late_rate`：实际送达日期晚于预计送达日期的比例，仅对已送达且有送达日期的订单计算
- `review_score_mean`：订单评价分数均值
- `low_review_rate`：评分 <= 2 的订单比例


In [ ]:
delivered = orders[orders["is_delivered"] & orders["delivery_days"].notna()].copy()
reviewed = orders[orders["review_score_mean"].notna()].copy()
paid = orders[orders["payment_total"].notna()].copy()

kpis = pd.DataFrame({
    "metric": [
        "total_orders",
        "delivered_orders",
        "delivery_rate",
        "gmv_payment_total",
        "product_total",
        "freight_total",
        "aov",
        "freight_share_of_payment",
        "avg_delivery_days",
        "late_rate_delivered",
        "avg_review_score",
        "low_review_rate",
    ],
    "value": [
        len(orders),
        len(delivered),
        len(delivered) / len(orders),
        paid["payment_total"].sum(),
        orders["product_total"].sum(),
        orders["freight_total"].sum(),
        paid["payment_total"].mean(),
        orders["freight_total"].sum() / paid["payment_total"].sum(),
        delivered["delivery_days"].mean(),
        delivered["is_late"].mean(),
        reviewed["review_score_mean"].mean(),
        reviewed["is_low_review"].mean(),
    ],
})

kpis


## 3. 月度增长趋势

先看平台整体增长：订单量、GMV、客单价。真实业务里，这通常是最先放在汇报首页的部分。

In [ ]:
monthly = (
    paid
    .groupby("purchase_month_dt", as_index=False)
    .agg(
        orders=("order_id", "count"),
        gmv=("payment_total", "sum"),
        product_total=("product_total", "sum"),
        freight_total=("freight_total", "sum"),
        aov=("payment_total", "mean"),
        avg_review_score=("review_score_mean", "mean"),
    )
)

monthly["gmv_mom_growth"] = monthly["gmv"].pct_change()
monthly["orders_mom_growth"] = monthly["orders"].pct_change()
monthly.tail()


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

ax1.plot(monthly["purchase_month_dt"], monthly["orders"], marker="o", color="#1f77b4", label="Orders")
ax2.plot(monthly["purchase_month_dt"], monthly["gmv"], marker="o", color="#d62728", label="GMV")

ax1.set_title("Monthly Orders and GMV")
ax1.set_xlabel("Purchase Month")
ax1.set_ylabel("Orders")
ax2.set_ylabel("GMV")
ax1.tick_params(axis="x", rotation=45)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")
if ax2.get_legend() is not None:
    ax2.get_legend().remove()

plt.tight_layout()
plt.savefig(figures_dir / "monthly_orders_gmv.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["purchase_month_dt"], monthly["aov"], marker="o", color="#2ca02c")
ax.set_title("Monthly Average Order Value")
ax.set_xlabel("Purchase Month")
ax.set_ylabel("AOV")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(figures_dir / "monthly_aov.png", dpi=160, bbox_inches="tight")
plt.show()


## 4. 订单状态与履约漏斗

订单状态能帮助判断业务漏斗是否健康。大部分订单应处于 `delivered`，其他状态需要作为异常或未完成流程看待。

In [ ]:
status_summary = (
    orders["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="orders")
)
status_summary["order_share"] = status_summary["orders"] / status_summary["orders"].sum()
status_summary


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
status_plot = status_summary.sort_values("orders", ascending=True)
ax.barh(status_plot["order_status"], status_plot["orders"], color="#4c78a8")
ax.set_title("Order Status Distribution")
ax.set_xlabel("Orders")
ax.set_ylabel("Order Status")
plt.tight_layout()
plt.savefig(figures_dir / "order_status_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


## 5. 物流履约表现

物流是这个数据集最有业务价值的方向之一。这里关注配送时长、延迟率，以及它们和评价的关系。

In [ ]:
delivery_summary = delivered[["delivery_days", "estimated_delivery_days", "delivery_delta_days"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
delivery_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(delivered["delivery_days"].clip(upper=60).dropna(), bins=30, color="#4c78a8", edgecolor="white")
axes[0].set_title("Delivery Days Distribution")
axes[0].set_xlabel("Delivery Days (capped at 60)")

axes[1].hist(delivered["delivery_delta_days"].clip(lower=-30, upper=30).dropna(), bins=30, color="#f58518", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_title("Actual vs Estimated Delivery Delta")
axes[1].set_xlabel("Actual Delivery Date - Estimated Date")

plt.tight_layout()
plt.savefig(figures_dir / "delivery_distributions.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
monthly_delivery = (
    delivered
    .groupby("purchase_month_dt", as_index=False)
    .agg(
        orders=("order_id", "count"),
        avg_delivery_days=("delivery_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review_score=("review_score_mean", "mean"),
    )
)

monthly_delivery.tail()


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

ax1.plot(monthly_delivery["purchase_month_dt"], monthly_delivery["avg_delivery_days"], marker="o", color="#1f77b4", label="Avg Delivery Days")
ax2.plot(monthly_delivery["purchase_month_dt"], monthly_delivery["late_rate"], marker="o", color="#d62728", label="Late Rate")

ax1.set_title("Monthly Delivery Performance")
ax1.set_xlabel("Purchase Month")
ax1.set_ylabel("Average Delivery Days")
ax2.set_ylabel("Late Rate")
ax1.tick_params(axis="x", rotation=45)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")
if ax2.get_legend() is not None:
    ax2.get_legend().remove()

plt.tight_layout()
plt.savefig(figures_dir / "monthly_delivery_performance.png", dpi=160, bbox_inches="tight")
plt.show()


## 6. 评价与客户体验

评价分数是客户体验的近似指标。这里重点看低评分订单，以及延迟配送是否和低评分有关。

In [ ]:
review_distribution = (
    reviewed["review_score_min"]
    .round()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="orders")
)
review_distribution["order_share"] = review_distribution["orders"] / review_distribution["orders"].sum()
review_distribution


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(review_distribution["review_score"], review_distribution["orders"], color="#54a24b")
ax.set_title("Review Score Distribution")
ax.set_xlabel("Review Score")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.savefig(figures_dir / "review_score_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
delay_review = (
    delivered[delivered["review_score_mean"].notna()]
    .assign(delivery_bucket=pd.cut(
        delivered["delivery_delta_days"],
        bins=[-np.inf, -7, 0, 3, 7, 14, np.inf],
        labels=["7+ days early", "On time/early", "1-3 days late", "4-7 days late", "8-14 days late", "15+ days late"],
    ))
    .groupby("delivery_bucket", observed=True)
    .agg(
        orders=("order_id", "count"),
        avg_review_score=("review_score_mean", "mean"),
        low_review_rate=("is_low_review", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
    )
    .reset_index()
)

delay_review


In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

delay_x = np.arange(len(delay_review))
ax1.bar(delay_x, delay_review["orders"], color="#bab0ac", label="Orders")
ax2.plot(delay_x, delay_review["low_review_rate"], marker="o", color="#d62728", label="Low Review Rate")

ax1.set_title("Delivery Delay and Low Review Rate")
ax1.set_xlabel("Delivery Timing Bucket")
ax1.set_ylabel("Orders")
ax2.set_ylabel("Low Review Rate")
ax1.set_xticks(delay_x)
ax1.set_xticklabels(delay_review["delivery_bucket"], rotation=25, ha="right")

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left")
if ax2.get_legend() is not None:
    ax2.get_legend().remove()

plt.tight_layout()
plt.savefig(figures_dir / "delivery_delay_low_review_rate.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. 品类表现

品类分析不只看 GMV，还要同时看订单量、客单价、运费占比、延迟率和评分。高收入但低体验的品类，通常是运营优先级最高的地方。

In [ ]:
category_summary = (
    orders[orders["main_product_category"].notna()]
    .groupby("main_product_category", as_index=False)
    .agg(
        orders=("order_id", "count"),
        gmv=("payment_total", "sum"),
        product_total=("product_total", "sum"),
        freight_total=("freight_total", "sum"),
        aov=("payment_total", "mean"),
        avg_review_score=("review_score_mean", "mean"),
        low_review_rate=("is_low_review", "mean"),
        late_rate=("is_late", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
    )
)
category_summary["freight_share"] = category_summary["freight_total"] / category_summary["gmv"]
category_summary = category_summary.sort_values("gmv", ascending=False)
category_summary.head(15)


In [ ]:
top_categories = category_summary.head(15).copy()

fig, ax = plt.subplots(figsize=(12, 7))
top_categories_plot = top_categories.sort_values("gmv", ascending=True)
ax.barh(top_categories_plot["main_product_category"], top_categories_plot["gmv"], color="#4c78a8")
ax.set_title("Top 15 Categories by GMV")
ax.set_xlabel("GMV")
ax.set_ylabel("Category")
plt.tight_layout()
plt.savefig(figures_dir / "top_categories_by_gmv.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
category_risk = category_summary[category_summary["orders"] >= 300].copy()
category_risk = category_risk.sort_values(["low_review_rate", "gmv"], ascending=[False, False])
category_risk.head(15)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
plot_data = category_summary[category_summary["orders"] >= 300].copy()

size_min, size_max = 40, 700
gmv_scaled = (plot_data["gmv"] - plot_data["gmv"].min()) / (plot_data["gmv"].max() - plot_data["gmv"].min())
bubble_sizes = size_min + gmv_scaled.fillna(0) * (size_max - size_min)
scatter = ax.scatter(
    plot_data["late_rate"],
    plot_data["avg_review_score"],
    s=bubble_sizes,
    c=plot_data["freight_share"],
    cmap="viridis_r",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.6,
)
fig.colorbar(scatter, ax=ax, label="Freight Share")

ax.set_title("Category Experience Map: Late Rate vs Review Score")
ax.set_xlabel("Late Rate")
ax.set_ylabel("Average Review Score")

for _, row in plot_data.sort_values("gmv", ascending=False).head(8).iterrows():
    ax.text(row["late_rate"], row["avg_review_score"], row["main_product_category"], fontsize=8, alpha=0.8)

plt.tight_layout()
plt.savefig(figures_dir / "category_experience_map.png", dpi=160, bbox_inches="tight")
plt.show()


## 8. 地区表现

地区分析帮助判断增长是否集中在少数州，以及不同地区的履约体验是否有明显差异。

In [ ]:
state_summary = (
    orders[orders["customer_state"].notna()]
    .groupby("customer_state", as_index=False)
    .agg(
        orders=("order_id", "count"),
        gmv=("payment_total", "sum"),
        aov=("payment_total", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_review_score=("review_score_mean", "mean"),
        low_review_rate=("is_low_review", "mean"),
    )
)
state_summary["gmv_share"] = state_summary["gmv"] / state_summary["gmv"].sum()
state_summary = state_summary.sort_values("gmv", ascending=False)
state_summary.head(15)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

state_top_plot = state_summary.head(15).sort_values("gmv", ascending=True)
axes[0].barh(state_top_plot["customer_state"], state_top_plot["gmv"], color="#4c78a8")
axes[0].set_title("Top States by GMV")
axes[0].set_xlabel("GMV")
axes[0].set_ylabel("Customer State")

state_size_min, state_size_max = 50, 700
state_scaled = (state_summary["orders"] - state_summary["orders"].min()) / (state_summary["orders"].max() - state_summary["orders"].min())
state_sizes = state_size_min + state_scaled.fillna(0) * (state_size_max - state_size_min)
state_scatter = axes[1].scatter(
    state_summary["avg_delivery_days"],
    state_summary["avg_review_score"],
    s=state_sizes,
    c=state_summary["late_rate"],
    cmap="magma_r",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.6,
)
fig.colorbar(state_scatter, ax=axes[1], label="Late Rate")
axes[1].set_title("State Delivery Experience")
axes[1].set_xlabel("Average Delivery Days")
axes[1].set_ylabel("Average Review Score")

for _, row in state_summary.head(8).iterrows():
    axes[1].text(row["avg_delivery_days"], row["avg_review_score"], row["customer_state"], fontsize=9)

plt.tight_layout()
plt.savefig(figures_dir / "state_revenue_delivery_experience.png", dpi=160, bbox_inches="tight")
plt.show()


## 9. 支付方式与分期

支付方式会影响客单价、转化路径和风险控制。这里先做基础分布分析。

In [ ]:
payment_summary = (
    orders[orders["primary_payment_type"].notna()]
    .groupby("primary_payment_type", as_index=False)
    .agg(
        orders=("order_id", "count"),
        gmv=("payment_total", "sum"),
        aov=("payment_total", "mean"),
        avg_installments=("max_payment_installments", "mean"),
        avg_review_score=("review_score_mean", "mean"),
    )
)
payment_summary["order_share"] = payment_summary["orders"] / payment_summary["orders"].sum()
payment_summary = payment_summary.sort_values("orders", ascending=False)
payment_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

payment_orders_plot = payment_summary.sort_values("orders", ascending=True)
axes[0].barh(payment_orders_plot["primary_payment_type"], payment_orders_plot["orders"], color="#4c78a8")
axes[0].set_title("Orders by Primary Payment Type")
axes[0].set_xlabel("Orders")
axes[0].set_ylabel("Payment Type")

payment_aov_plot = payment_summary.sort_values("aov", ascending=True)
axes[1].barh(payment_aov_plot["primary_payment_type"], payment_aov_plot["aov"], color="#f58518")
axes[1].set_title("AOV by Primary Payment Type")
axes[1].set_xlabel("AOV")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(figures_dir / "payment_type_summary.png", dpi=160, bbox_inches="tight")
plt.show()


## 10. 自动生成关键发现

下面这段会把主要发现整理成 Markdown，保存到 `reports/eda_key_findings.md`。后续写 README 或项目报告时可以直接引用，并结合业务含义进行解释。

In [ ]:
def pct(x):
    return f"{x:.1%}"

def money(x):
    return f"{x:,.0f} BRL"

best_month = monthly.loc[monthly["gmv"].idxmax()]
top_category = category_summary.iloc[0]
top_state = state_summary.iloc[0]
worst_delay_bucket = delay_review.sort_values("low_review_rate", ascending=False).iloc[0]
largest_payment_type = payment_summary.iloc[0]

findings = f"""# EDA Key Findings

## Executive KPIs

- Total orders: {len(orders):,}
- Delivered orders: {len(delivered):,} ({pct(len(delivered) / len(orders))})
- GMV based on payment total: {money(paid['payment_total'].sum())}
- Average order value: {money(paid['payment_total'].mean())}
- Freight share of payment total: {pct(orders['freight_total'].sum() / paid['payment_total'].sum())}
- Average delivery time: {delivered['delivery_days'].mean():.1f} days
- Late delivery rate among delivered orders: {pct(delivered['is_late'].mean())}
- Average review score: {reviewed['review_score_mean'].mean():.2f}
- Low review rate (score <= 2): {pct(reviewed['is_low_review'].mean())}

## Growth

- Highest GMV month: {best_month['purchase_month_dt'].strftime('%Y-%m')} with {money(best_month['gmv'])} GMV and {int(best_month['orders']):,} orders.

## Category

- Top category by GMV: {top_category['main_product_category']} with {money(top_category['gmv'])} GMV across {int(top_category['orders']):,} orders.

## Geography

- Top customer state by GMV: {top_state['customer_state']} with {money(top_state['gmv'])} GMV and {int(top_state['orders']):,} orders.

## Customer Experience

- The highest low-review bucket is `{worst_delay_bucket['delivery_bucket']}` with low review rate of {pct(worst_delay_bucket['low_review_rate'])}.
- Largest payment type by order count: {largest_payment_type['primary_payment_type']} with {int(largest_payment_type['orders']):,} orders.

## Business Interpretation

A useful next step is to quantify how delivery delay, freight burden, product category, and region relate to low-review risk. I use that direction in the modeling notebook.
"""

report_path = project_root / "reports" / "eda_key_findings.md"
report_path.write_text(findings, encoding="utf-8")
print(findings)
print(f"Saved to: {report_path}")


## 11. 本阶段结论

本阶段产出了一版完整商业 EDA：

- 总体 KPI
- 月度增长趋势
- 订单状态和履约漏斗
- 物流配送表现
- 评分和低评分风险
- 品类表现
- 地区表现
- 支付方式结构

下一步是 SQL 分析层：把核心问题改写成 SQL 查询，让项目同时体现 Python 和 SQL 能力。之后再做 Dashboard 或机器学习模型。